# The EM algorithm for the admixture model

In this exercise we will open up the black box of `ADMIXTURE`/`NGSadmix` and implement the
EM algorithm that estimates the admixture proportions **Q** and the ancestral allele
frequencies **F**.

**Learning objectives**

 - Be able to write up the likelihood of the admixture model
 - Understand which quantity is the hidden (latent) variable in the model
 - Be able to perform the E-step: the probability that an allele copy comes from a given ancestral population
 - Be able to perform the M-step: the update of **Q** and **F**
 - Implement a complete EM algorithm and see it converge to the maximum likelihood estimate

The exercise follows the slides from the admixture lecture. Everything is done in R on
simulated data, so you can see how close the estimates get to the truth.

# 1. The model

We have $N$ individuals typed at $M$ SNPs and we assume there are $K$ ancestral populations.

 - $Q$ is an $N \times K$ matrix. $Q_{ik}$ is the proportion of individual $i$'s genome
   that comes from population $k$, so $\sum_k Q_{ik}=1$.
 - $F$ is an $M \times K$ matrix. $F_{jk}$ is the frequency of the (say) `A` allele at
   SNP $j$ in population $k$. On the slides this is sometimes called $P$ ("Phrequencies").
 - $G_{ij} \in \{0,1,2\}$ is the observed genotype: the number of `A` alleles that
   individual $i$ carries at SNP $j$. We write the two allele copies as $G_{ij1}$ and
   $G_{ij2}$, so $G_{ij}=G_{ij1}+G_{ij2}$.

Each allele copy is drawn in two steps

 1. pick an ancestral population $A \in \{1,\dots,K\}$ with probability $p(A=k\mid Q^i)=Q_{ik}$
 2. draw the allele from that population with probability $p(G_{ijd}=\text{A}\mid F^j, A=k)=F_{jk}$

So the probability that one allele copy of individual $i$ at SNP $j$ is an `A` allele is the
**individual allele frequency**

$$ h_{ij} \;=\; p(G_{ijd}=\text{A} \mid Q^i,F^j)\;=\;\sum_{k=1}^{K} Q_{ik}F_{jk} $$

**Questions**

 - Why do we sum over $k$? Which variable are we summing out?
 - Assume Hardy-Weinberg equilibrium *within* the individual. Write up $p(G_{ij}=1\mid Q^i,F^j)$ using $h_{ij}$.
 - With $K=2$, $N=5$ individuals and $M=8$ SNPs, how many parameters does the model have?
   (the general answer is on the slides: $M\times K + N \times K$, but how many are *free*?)

Let us redo the small calculation from the slides. Individual $i$ has admixture proportions
$Q_i=(0.4,0.6)$ and at SNP $j$ the frequency of the `T` allele is $0.7$ in population 1 and
$0.2$ in population 2. The individual has the genotype `TT`.

In [ ]:
Qi <- c(0.4, 0.6)   # admixture proportions of the individual
Fj <- c(0.7, 0.2)   # frequency of the T allele in pop 1 and pop 2

## individual allele frequency: probability that ONE allele copy is a T
h <- sum(Qi * Fj)
cat("h = p(allele = T) =", h, "\n")

## the genotype is TT, i.e. both allele copies are T
cat("p(G = TT) =", h^2, "\n")

 - Recalculate $p(G=\text{TT})$ by hand and check that you get the same.
 - What is $p(G=\text{TC})$ (one T and one C allele)?
 - What would $p(G=\text{TT})$ be if the individual was not admixed but came entirely from population 2?

# 2. The likelihood

Assuming that the SNPs are independent and that the two allele copies of an individual are
independent (HWE), the likelihood of the whole data set is (slide *EM algorithm - reformulate the likelihood*)

$$
p(G\mid Q,F)=\prod_i^N\prod_j^M p(G_{ij}\mid Q^i,F^j)
\;\propto\; \prod_i^N\prod_j^M\prod_{d}^{2} p(G_{ijd}\mid Q^i,F^j)
$$

and by introducing the hidden ancestry state $A$ of each allele copy

$$
=\prod_i^N\prod_j^M\prod_{d}^{2}\sum_{A} p(G_{ijd}\mid F^j,A)\,p(A\mid Q^i)
$$

## The likelihood written with the individual allele frequency

Once we have summed out the hidden ancestry state, each of the two allele copies of
individual $i$ at SNP $j$ is simply an `A` allele with probability $h_{ij}=\sum_k Q_{ik}F_{jk}$,
independently of each other (that is the HWE assumption). Two independent draws with the same
success probability is exactly a **binomial distribution** with $n=2$ trials

$$ G_{ij} \mid Q^i,F^j \;\sim\; \text{Binomial}(2, h_{ij}) $$

so

$$
p(G_{ij}\mid Q^i,F^j)=\binom{2}{G_{ij}}\,h_{ij}^{G_{ij}}\,(1-h_{ij})^{2-G_{ij}}
$$

which written out for the three possible genotypes is just HWE with the frequency $h_{ij}$

| $G_{ij}$ | genotype | $p(G_{ij}\mid Q^i,F^j)$ |
|---|---|---|
| 0 | no `A` alleles | $(1-h_{ij})^2$ |
| 1 | heterozygote | $2\,h_{ij}(1-h_{ij})$ |
| 2 | two `A` alleles | $h_{ij}^2$ |

Note that this is the *usual* HWE formula - the only new thing is that every individual has
its **own** allele frequency $h_{ij}$ at each SNP, given by its admixture proportions.

Taking the logarithm of the binomial probability and summing over all individuals and SNPs
gives the log likelihood

$$
\log p(G\mid Q,F) \;=\; \sum_{i}^{N}\sum_{j}^{M}
\Big[ G_{ij}\log h_{ij} + (2-G_{ij})\log (1-h_{ij}) \Big] + \text{constant}
$$

The constant is $\sum_{ij}\log\binom{2}{G_{ij}}$. It does not depend on $Q$ or $F$, so we can
drop it - it shifts the log likelihood up or down but does not move the maximum.

Read the two terms as counts: $G_{ij}$ is the number of `A` alleles the individual carries
and $2-G_{ij}$ is the number of other alleles, each weighted by the log probability of
drawing it.

**Questions**

 - Which term in the sum $\sum_A p(G_{ijd}|F^j,A)p(A|Q^i)$ is $F$ and which is $Q$?
 - Write up $\log p(G_{ij}|Q^i,F^j)$ for a heterozygote and check that it matches the formula above.
 - Why are we allowed to throw away the binomial coefficient $\binom{2}{G_{ij}}$?
 - Why is $\sum_k Q_{ik}F_{jk}$ *not* the allele frequency of a population, but of an individual?
 - Why is there no analytical solution (i.e. why can we not just set the derivative to zero and solve)?

## Simulate some data

We simulate $N=50$ individuals and $M=2000$ SNPs from $K=2$ populations. The first 20
individuals are from population 1, the last 20 from population 2 and the 10 in the middle
are admixed. The ancestral frequencies are drawn with a Balding-Nichols model with
$F_{ST}=0.15$, i.e. the two populations are fairly differentiated.

In [ ]:
set.seed(3)
N <- 50      # individuals
M <- 2000    # SNPs
K <- 2       # ancestral populations
Fst <- 0.15  # how differentiated the two populations are

## true ancestral allele frequencies (M x K)
anc <- runif(M, 0.05, 0.95)                        # frequency in the common ancestor
bn <- function(p, Fst) rbeta(length(p), p*(1-Fst)/Fst, (1-p)*(1-Fst)/Fst)
Ftrue <- cbind(bn(anc, Fst), bn(anc, Fst))

## true admixture proportions (N x K)
q1 <- c(rep(1, 20), seq(0.9, 0.1, length.out = 10), rep(0, 20))
Qtrue <- cbind(q1, 1 - q1)

## simulate the genotypes: G_ij ~ Binomial(2, h_ij)
Htrue <- Qtrue %*% t(Ftrue)                        # N x M individual allele frequencies
G <- matrix(rbinom(N*M, 2, Htrue), nrow = N, ncol = M)

cat("dimension of the genotype matrix:", dim(G), "\n")
print(G[1:5, 1:8])

In [ ]:
## the true admixture proportions we are trying to recover
barplot(t(Qtrue), col = c("steelblue", "red"), border = NA, space = 0,
        xlab = "individual", ylab = "admixture proportion", main = "TRUE Q")

Now the log likelihood function. Note that it only needs `Q` and `F` through the individual
allele frequencies `H`.

In [ ]:
logLike <- function(Q, F, G){
  H <- Q %*% t(F)                                  # N x M individual allele frequencies
  sum( G * log(H) + (2 - G) * log(1 - H) )
}

cat("log likelihood at the true parameters:", logLike(Qtrue, Ftrue, G), "\n")
cat("log likelihood at a random guess     :",
    logLike(matrix(0.5, N, K), matrix(0.5, M, K), G), "\n")

 - Identify $h_{ij}$, $G_{ij}\log h_{ij}$ and $(2-G_{ij})\log(1-h_{ij})$ in the code.
 - Why is the likelihood higher for the true parameters?

# 3. The E-step

The EM algorithm works because we can introduce the hidden variable $A$ - the ancestral
population that an allele copy came from. From the slides (*EM algorithm - Fits our problem*)
the E-step is the posterior probability of the hidden state

$$
\mathbf{q_i}(A_z)=p(A_z\mid G_{ijd},Q^{(n)},F^{(n)})
$$

which by Bayes' formula is

$$
p(A=k\mid G_{ijd},F_j,Q_i)=
\frac{p(G_{ijd}\mid A=k,F_j)\,p(A=k\mid Q_i)}{\sum_{k'}p(G_{ijd}\mid A=k',F_j)\,p(A=k'\mid Q_i)}
= \frac{F_{jk}^{G_{ijd}}Q_{ik}}{\sum_{k'}F_{jk'}^{G_{ijd}}Q_{ik'}}
$$

where $F_{jk}^{G_{ijd}}$ is shorthand for "$F_{jk}$ if the allele copy is an `A` allele and
$1-F_{jk}$ if it is not".

Let us reproduce the numbers on the slide. Individual 1 has $Q_i=(0.2,0.8)$ and at SNP 1 the
frequency of the `A` allele is $0.2$ in population 1 and $0.3$ in population 2. The genotype
of the individual is `AT`, i.e. one allele copy is an `A` and the other is a `T`.

In [ ]:
Qi <- c(0.2, 0.8)    # admixture proportions of individual 1
Fj <- c(0.2, 0.3)    # frequency of the A allele at SNP 1 in the two populations

## allele copy 1 is an A allele
pA <- Fj * Qi / sum(Fj * Qi)
## allele copy 2 is a T allele  (probability 1-F of being drawn from each population)
pT <- (1 - Fj) * Qi / sum((1 - Fj) * Qi)

cat("p(ancestry | the A allele) :", round(pA, 2), "\n")
cat("p(ancestry | the T allele) :", round(pT, 2), "\n")

 - Compare with the table on the slides (0.14/0.86 and 0.22/0.78). Do you get the same?
 - The individual has 80% ancestry from population 2 but the `A` allele is assigned to
   population 2 with 86%. Why is it higher than 80%?
 - What happens to the posterior if the two populations have the same allele frequency
   ($F_{j1}=F_{j2}$)? Try it in the code above. What does that tell you about which SNPs
   carry information about ancestry?
 - What happens if the individual is not admixed, $Q_i=(1,0)$?

# 4. The M-step

In the M-step we update the parameters using the posterior probabilities as if they were
observed counts. From the slides

$$\theta^{(n+1)}_z=\frac{\sum_i \mathbf{q_i}(A_z)}{\sum_i\sum_z \mathbf{q_i}(A_j)}$$

For our model it is convenient to define, for each individual $i$, SNP $j$ and population $k$,
the **expected number of allele copies coming from population $k$**. Since a genotype
$G_{ij}$ contains $G_{ij}$ `A` alleles and $2-G_{ij}$ other alleles

$$
a_{ijk}= G_{ij}\;\frac{Q_{ik}F_{jk}}{h_{ij}}
\qquad\qquad
b_{ijk}= (2-G_{ij})\;\frac{Q_{ik}(1-F_{jk})}{1-h_{ij}}
$$

$a_{ijk}$ is the expected number of `A` alleles from population $k$ and $b_{ijk}$ the
expected number of non-`A` alleles from population $k$.

The updates are then just the corresponding frequencies

$$
Q^{(n+1)}_{ik}=\frac{\sum_j (a_{ijk}+b_{ijk})}{2M}
\qquad\qquad
F^{(n+1)}_{jk}=\frac{\sum_i a_{ijk}}{\sum_i (a_{ijk}+b_{ijk})}
$$

**Questions**

 - $Q_{ik}$ is updated by summing over SNPs while $F_{jk}$ is updated by summing over
   individuals. Explain why.
 - Why do we divide by $2M$ in the update of $Q$?
 - Convince yourself that $\sum_k Q^{(n+1)}_{ik}=1$ automatically. (hint: what is
   $\sum_k a_{ijk}$?)
 - Compare the two updates with the simple allele frequency estimator you have seen before:
   "expected number of A alleles divided by expected number of alleles". Where is the
   difference?

## A note on `outer()`

The code below is written with matrices so that it runs fast. The only trick used is
`outer()`, which is the **outer product** of two vectors: it multiplies every element of the
first vector with every element of the second

$$
\mathrm{outer}(x,y) = x\,y^{T} =
\begin{pmatrix} x_1 \\ x_2 \\ \vdots \\ x_N \end{pmatrix}
\begin{pmatrix} y_1 & y_2 & \cdots & y_M \end{pmatrix}
=
\begin{pmatrix}
x_1y_1 & x_1y_2 & \cdots & x_1y_M \\
x_2y_1 & x_2y_2 & \cdots & x_2y_M \\
\vdots & \vdots & \ddots & \vdots \\
x_Ny_1 & x_Ny_2 & \cdots & x_Ny_M
\end{pmatrix}
$$

so element $(i,j)$ of `outer(x, y)` is simply $x_i y_j$. With $x=Q_{\cdot k}$ (a vector of
length $N$) and $y=F_{\cdot k}$ (a vector of length $M$) we get the $N\times M$ matrix that
holds $Q_{ik}F_{jk}$ for **all** individuals and SNPs at once - the contribution of
population $k$ to the individual allele frequency.

The individual allele frequencies themselves are the matrix product
$H = Q F^{T}$, i.e. $h_{ij}=\sum_k Q_{ik}F_{jk}$, which in R is `Q %*% t(F)`. In other words

$$ H \;=\; Q F^{T} \;=\; \sum_{k=1}^{K} \mathrm{outer}(Q_{\cdot k}, F_{\cdot k}) $$

The matrix product sums over the $K$ populations, while `outer` keeps one population at a
time - which is exactly what we need in the E-step, where we want the ancestry probabilities
separately for each population.

In [ ]:
x <- c(1, 2, 3)        # imagine this is Q[,k]  (N = 3 individuals)
y <- c(10, 20)         # imagine this is F[,k]  (M = 2 SNPs)

outer(x, y)            # 3 x 2 matrix with element (i,j) = x_i * y_j
x %*% t(y)             # exactly the same thing written as a matrix product

## and the sum over the K populations is the matrix product Q %*% t(F)
H1 <- Qtrue %*% t(Ftrue)
H2 <- outer(Qtrue[,1], Ftrue[,1]) + outer(Qtrue[,2], Ftrue[,2])
cat("are the two ways of getting H identical?", all.equal(H1, H2), "\n")

 - Check by hand that `outer(c(1,2,3), c(10,20))` gives what you expect.
 - Why does `H` have dimensions $N \times M$ and not $N \times K$ or $M \times K$?

In [ ]:
## one full EM step: returns the updated Q and F
emStep <- function(Q, F, G){
  N <- nrow(Q); M <- nrow(F); K <- ncol(Q)
  H <- Q %*% t(F)                                   # N x M individual allele frequencies

  Qnew <- matrix(0, N, K)
  Fnum <- matrix(0, M, K)     # numerator   sum_i a_ijk
  Fden <- matrix(0, M, K)     # denominator sum_i (a_ijk + b_ijk)

  for(k in 1:K){
    ## E-step: expected number of allele copies from population k
    a <- G       * outer(Q[,k],     F[,k])  / H       # the A alleles
    b <- (2 - G) * outer(Q[,k], 1 - F[,k])  / (1 - H) # the other alleles

    ## M-step
    Qnew[,k] <- rowSums(a + b) / (2 * M)              # sum over SNPs
    Fnum[,k] <- colSums(a)                            # sum over individuals
    Fden[,k] <- colSums(a + b)
  }
  ## keep the frequencies away from 0 and 1 so log(0) does not appear in the likelihood
  bound <- function(x, e = 1e-5) pmin(pmax(x, e), 1 - e)
  list(Q = Qnew, F = bound(Fnum / Fden))
}

 - Find the E-step and the M-step in the function.
 - `outer(Q[,k], F[,k])` is the $N\times M$ matrix with entries $Q_{ik}F_{jk}$ and `H` is the
   matrix with entries $h_{ij}$. Which part of the formula is `outer(Q[,k], F[,k]) / H`?
 - Why is `a` multiplied by `G` and `b` by `2-G`?
 - `rowSums` sums over SNPs and `colSums` over individuals. Which one belongs to $Q$ and
   which to $F$?

# 5. The complete EM algorithm

An EM algorithm is just: start at a random guess, do E- and M-steps until the likelihood
stops increasing.

In [ ]:
admixEM <- function(G, K, maxIter = 500, tol = 0.1, seed = 1){
  set.seed(seed)
  N <- nrow(G); M <- ncol(G)

  ## random starting point
  Q <- matrix(runif(N*K), N, K); Q <- Q / rowSums(Q)   # rows must sum to one
  F <- matrix(runif(M*K, 0.1, 0.9), M, K)

  ll <- logLike(Q, F, G)
  llTrace <- ll
  for(iter in 1:maxIter){
    par <- emStep(Q, F, G)
    Q <- par$Q
    F <- par$F
    llNew <- logLike(Q, F, G)
    llTrace <- c(llTrace, llNew)
    if(abs(llNew - ll) < tol){                         # converged: the likelihood barely moves
      cat("converged after", iter, "iterations\n")
      break
    }
    ll <- llNew
  }
  if(iter == maxIter)
    cat("stopped after", maxIter, "iterations without reaching the tolerance\n")
  list(Q = Q, F = F, logLike = ll, trace = llTrace, iterations = iter)
}

res <- admixEM(G, K = 2)
cat("log likelihood of the estimate:", res$logLike, "\n")
cat("log likelihood of the truth   :", logLike(Qtrue, Ftrue, G), "\n")

In [ ]:
## the likelihood must increase in every single iteration
plot(res$trace, type = "l", xlab = "EM iteration", ylab = "log likelihood")
abline(h = logLike(Qtrue, Ftrue, G), col = "red", lty = 2)
legend("bottomright", c("EM", "true parameters"), col = c("black", "red"), lty = c(1, 2))

 - Does the likelihood increase in every iteration? It is guaranteed to by the EM algorithm.
 - The estimate can have a **higher** likelihood than the true parameters. Is that a bug?
   (hint: how many parameters are we estimating compared to how much data we have?)
 - Look at the curve: the likelihood jumps a lot in the first iterations and then creeps.
   The EM algorithm is very stable but converges slowly, so we stop when the log likelihood
   changes by less than `tol = 0.1` between two iterations rather than waiting for it to stop
   changing completely. Real implementations use accelerated versions of the EM algorithm
   (`ADMIXTURE` and `NGSadmix` both do). Try
   `plot(admixEM(G, K = 2, maxIter = 20)$Q[,1], res$Q[,1])` - are 20 iterations enough?
 - Try a much stricter tolerance, e.g. `admixEM(G, K = 2, tol = 1e-6)`. How many more
   iterations does it use, and does the estimate of $Q$ change much?

In [ ]:
par(mfrow = c(2,1), mar = c(4,4,2,1))
barplot(t(Qtrue), col = c("steelblue","red"), border = NA, space = 0,
        ylab = "proportion", main = "TRUE Q")
barplot(t(res$Q),  col = c("steelblue","red"), border = NA, space = 0,
        ylab = "proportion", xlab = "individual", main = "ESTIMATED Q")

In [ ]:
par(mfrow = c(1,2))
plot(Qtrue[,1], res$Q[,1], xlab = "true Q (pop 1)", ylab = "estimated Q (pop 1)",
     main = "admixture proportions"); abline(0, 1, col = "red")
plot(Ftrue[,1], res$F[,1], xlab = "true F (pop 1)", ylab = "estimated F (pop 1)",
     main = "allele frequencies", pch = 16, cex = 0.4, col = "#00000044")
abline(0, 1, col = "red")

 - How well are the admixture proportions estimated? And the allele frequencies?
 - It can happen that the colours are swapped, so that estimated population 1 corresponds to
   true population 2 and the points fall on the *other* diagonal. Why can the EM algorithm
   not know which population is "population 1"? This is called **label switching**.
 - $F$ has $M\times K=4000$ parameters and $Q$ has $N\times K=100$. Which of the two do you
   expect to be estimated most precisely, and why?

# 6. Things to be aware of

## Local optima
The likelihood surface of the admixture model has multiple maxima. The EM algorithm is only
guaranteed to climb to *a* maximum - not the global one. Therefore both `ADMIXTURE` and
`NGSadmix` are run several times with different starting points and the run with the highest
likelihood is kept.

In [ ]:
## run the EM 5 times from different random starting points
ll <- sapply(1:5, function(s) admixEM(G, K = 2, seed = s)$logLike)
print(ll)

 - Do all the runs end up at the same likelihood?
 - If two runs have very different likelihoods, which one would you keep?
 - In practice you would run it ~10 times and sort them by likelihood. If the top 5 have
   nearly the same likelihood you keep the top one, otherwise you run 10 more. Why is this a
   reasonable strategy?

## Choice of K
Try to run the algorithm with $K=3$ on data that was simulated with $K=2$.

In [ ]:
res3 <- admixEM(G, K = 3)
cat("K=2 log likelihood:", res$logLike,  "\n")
cat("K=3 log likelihood:", res3$logLike, "\n")

barplot(t(res3$Q), col = c("steelblue","red","darkgreen"), border = NA, space = 0,
        ylab = "proportion", xlab = "individual", main = "ESTIMATED Q with K=3")

 - Is the likelihood with $K=3$ higher or lower than with $K=2$? Will it always be that way?
 - Can you use the likelihood to choose $K$?
 - What does the third population look like - does it correspond to anything real?

# Bonus: from genotypes to genotype likelihoods (NGSadmix)

With low depth NGS data we do not know the genotypes. Instead of the called genotype we have
the genotype likelihoods $p(X_{ij}\mid G_{ij})$ for the three possible genotypes, and the
likelihood becomes (slide *NGSadmix*)

$$
p(X\mid Q,F)=\prod_i^N\prod_j^M \sum_{G_{ij}\in\{0,1,2\}} p(X_{ij}\mid G_{ij})\,p(G_{ij}\mid Q^i,F^j)
$$

where $p(G_{ij}\mid Q^i,F^j)$ is the HWE probability using the individual allele frequency
$h_{ij}$. The genotype is now *also* a hidden variable, so the E-step gets one extra layer:
first we get the posterior probability of the genotype, and from that the expected number of
`A` alleles

$$
E[G_{ij}] = \sum_{g\in\{0,1,2\}} g\; p(G_{ij}=g\mid X_{ij},Q^{(n)},F^{(n)})
$$

and then the M-step is **exactly the same as before** with $G_{ij}$ replaced by $E[G_{ij}]$.

In [ ]:
## simulate genotype likelihoods from 2x sequencing depth with an error rate of 1%
simGL <- function(G, depth = 2, e = 0.01){
  N <- nrow(G); M <- ncol(G)
  d <- matrix(rpois(N*M, depth), N, M)                 # number of reads
  pA <- c(1-e, 0.5, e)[G+1]                            # p(read is A | genotype)
  nA <- matrix(rbinom(N*M, d, pA), N, M)               # reads with the A allele
  ## genotype likelihoods p(X|G) for G = 0,1,2 copies of the A allele
  list(GL0 = dbinom(nA, d, e), GL1 = dbinom(nA, d, 0.5), GL2 = dbinom(nA, d, 1-e))
}

emStepGL <- function(Q, F, GL){
  N <- nrow(Q); M <- nrow(F); K <- ncol(Q)
  H <- Q %*% t(F)

  ## E-step part 1: posterior probability of the genotype (HWE prior with freq H)
  p0 <- GL$GL0 * (1-H)^2
  p1 <- GL$GL1 * 2*H*(1-H)
  p2 <- GL$GL2 * H^2
  tot <- p0 + p1 + p2
  EG <- (p1 + 2*p2) / tot                              # expected number of A alleles

  ## E-step part 2 + M-step: identical to before, with G replaced by EG
  Qnew <- matrix(0, N, K); Fnum <- matrix(0, M, K); Fden <- matrix(0, M, K)
  for(k in 1:K){
    a <- EG       * outer(Q[,k],     F[,k]) / H
    b <- (2 - EG) * outer(Q[,k], 1 - F[,k]) / (1 - H)
    Qnew[,k] <- rowSums(a + b) / (2 * M)
    Fnum[,k] <- colSums(a)
    Fden[,k] <- colSums(a + b)
  }
  bound <- function(x, e = 1e-5) pmin(pmax(x, e), 1 - e)
  list(Q = Qnew, F = bound(Fnum / Fden))
}

set.seed(7)
GL <- simGL(G, depth = 2)

## run the EM with the genotype likelihoods
set.seed(1)
Q <- matrix(runif(N*K), N, K); Q <- Q / rowSums(Q)
F <- matrix(runif(M*K, 0.1, 0.9), M, K)
for(i in 1:200){
  par <- emStepGL(Q, F, GL); Q <- par$Q; F <- par$F
}

par(mfrow = c(1,2))
barplot(t(Qtrue), col = c("steelblue","red"), border = NA, space = 0, main = "TRUE Q")
barplot(t(Q),     col = c("steelblue","red"), border = NA, space = 0, main = "NGSadmix-like Q (2x depth)")

 - Compare with the result from the called genotypes. How much do we lose by having only 2x depth?
 - Where in `emStepGL` is the genotype likelihood $p(X_{ij}\mid G_{ij})$ and where is the
   prior $p(G_{ij}\mid Q,F)$?
 - If you set the depth very high, `emStepGL` should give the same answer as `emStep`. Why?
 - Why is it a bad idea to call the genotypes first and then run the normal `ADMIXTURE`
   algorithm on low depth data?